In [1]:
import os
import math
import numpy as np
import pandas as pd

# =========================
# Config
# =========================
TARGET_COL = "Irrigation_Need"

CAT_COLS = [
    "Soil_Type",
    "Crop_Type",
    "Crop_Growth_Stage",
    "Season",
    "Irrigation_Type",
    "Water_Source",
    "Region",
]

OUT_DIR = "/kaggle/working/high_iv"
os.makedirs(OUT_DIR, exist_ok=True)

EPS = 0.5

# =========================
# IV / WOE
# =========================
def calc_binary_iv_for_categorical(
    df: pd.DataFrame,
    col: str,
    target_col: str,
    positive_label,
    eps: float = 0.5,
    treat_nan_as_str: bool = True,
):
    work = df[[col, target_col]].copy()

    if treat_nan_as_str:
        work[col] = work[col].astype("object").fillna("__MISSING__")
    else:
        work = work.dropna(subset=[col])

    bad_mask = (work[target_col] == positive_label)
    good_mask = ~bad_mask

    total_bad = int(bad_mask.sum())
    total_good = int(good_mask.sum())

    if total_bad == 0 or total_good == 0:
        raise ValueError(
            f"positive_label={positive_label} で片側しかありません。"
        )

    grouped = (
        work.groupby(col, dropna=False)[target_col]
        .agg(
            total_count="count",
            bad_count=lambda s: (s == positive_label).sum(),
            good_count=lambda s: (s != positive_label).sum(),
        )
        .reset_index()
        .rename(columns={col: "category"})
    )

    grouped["bad_count_s"] = grouped["bad_count"] + eps
    grouped["good_count_s"] = grouped["good_count"] + eps

    bad_total_s = grouped["bad_count_s"].sum()
    good_total_s = grouped["good_count_s"].sum()

    grouped["bad_dist"] = grouped["bad_count_s"] / bad_total_s
    grouped["good_dist"] = grouped["good_count_s"] / good_total_s

    grouped["woe"] = np.log(grouped["good_dist"] / grouped["bad_dist"])
    grouped["iv_component"] = (grouped["good_dist"] - grouped["bad_dist"]) * grouped["woe"]
    grouped["bad_rate_in_category"] = grouped["bad_count"] / grouped["total_count"]
    grouped["share_of_rows"] = grouped["total_count"] / len(work)
    grouped["positive_label"] = positive_label

    iv_value = float(grouped["iv_component"].sum())

    grouped = grouped.sort_values(
        ["iv_component", "total_count"],
        ascending=[False, False]
    ).reset_index(drop=True)

    return iv_value, grouped


def calc_multiclass_iv_table(
    df: pd.DataFrame,
    cat_cols: list[str],
    target_col: str,
    eps: float = 0.5,
):
    classes = sorted(df[target_col].dropna().unique().tolist())

    summary_rows = []
    detail_map = {}

    for col in cat_cols:
        per_class_iv = {}
        per_class_detail = []

        for cls in classes:
            iv_value, detail_df = calc_binary_iv_for_categorical(
                df=df,
                col=col,
                target_col=target_col,
                positive_label=cls,
                eps=eps,
            )
            per_class_iv[cls] = iv_value
            per_class_detail.append(detail_df)

        merged_detail = pd.concat(per_class_detail, axis=0, ignore_index=True)

        row = {
            "feature": col,
            "n_unique": df[col].nunique(dropna=False),
            "missing_count": df[col].isna().sum(),
            "iv_max": max(per_class_iv.values()),
            "iv_mean": float(np.mean(list(per_class_iv.values()))),
            "iv_sum": float(np.sum(list(per_class_iv.values()))),
            "best_class": max(per_class_iv, key=per_class_iv.get),
        }
        for cls, v in per_class_iv.items():
            row[f"iv_{cls}"] = v

        summary_rows.append(row)
        detail_map[col] = merged_detail

    summary = pd.DataFrame(summary_rows).sort_values(
        ["iv_max", "iv_mean"],
        ascending=[False, False]
    ).reset_index(drop=True)

    summary["iv_rank"] = np.arange(1, len(summary) + 1)
    return summary, detail_map

In [2]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")

iv_summary, iv_detail_map = calc_multiclass_iv_table(
    df=train,
    cat_cols=CAT_COLS,
    target_col=TARGET_COL,
    eps=EPS,
)

display(iv_summary)

iv_summary.to_csv(f"{OUT_DIR}/iv_summary_multiclass.csv", index=False)

for col, detail_df in iv_detail_map.items():
    detail_df.to_csv(f"{OUT_DIR}/iv_detail_{col}_multiclass.csv", index=False)

TOP_K = 5
high_iv_cols = iv_summary.head(TOP_K)["feature"].tolist()
print("high_iv_cols =", high_iv_cols)

,feature,n_unique,missing_count,iv_max,iv_mean,iv_sum,best_class,iv_High,iv_Low,iv_Medium,iv_rank
0,Crop_Growth_Stage,4,0,1.623334,1.424537,4.273610,High,1.623334,1.456510,1.193766,1
1,Water_Source,4,0,0.049047,0.019366,0.058097,High,0.049047,0.004643,0.004407,2
2,Crop_Type,6,0,0.043149,0.015603,0.046808,High,0.043149,0.001967,0.001693,3
3,Irrigation_Type,4,0,0.017273,0.009131,0.027393,High,0.017273,0.005592,0.004528,4
4,Soil_Type,4,0,0.014764,0.005459,0.016377,High,0.014764,0.001150,0.000463,5
5,Region,5,0,0.005520,0.002216,0.006648,High,0.005520,0.000624,0.000503,6
6,Season,3,0,0.002438,0.001928,0.005785,Low,0.001230,0.002438,0.002117,7


high_iv_cols = ['Crop_Growth_Stage', 'Water_Source', 'Crop_Type', 'Irrigation_Type', 'Soil_Type']
